In [1]:
import pandas as pd
import numpy as np
import re
import hashlib
from pathlib import Path

# Helper Functions

In [2]:
def stable_id(*parts: Any, prefix: str = '') -> str:
    raw = '||'.join('' if p is None else str(p) for p in parts)
    h = hashlib.md5(raw.encode('utf-8')).hexdigest()[:12]
    return f'{prefix}{h}' if prefix else h

def normalize_system_category(category: Any, text: str) -> str:
    c = category if pd.notna(category) else ''
    c_l = c.lower()
    if c_l in CATEGORY_MAP:
        return CATEGORY_MAP[c_l]
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            if label in ['turbo', 'intake']:
                return 'engine_power'
            return label
    return 'other'

def extract_subsystem(text: str) -> str:
    t = text.lower()
    for label, kws in PART_KEYWORDS.items():
        if any(k in t for k in kws):
            return label
    return 'other'

def extract_fitment_flags(text: str) -> dict[str, Any]:
    t = text.lower()
    years = [int(y) for y in re.findall(r'\b(20[12][0-9]|2030)\b', t)]
    
    # Filter years to valid A90 Supra range (2019-2026)
    valid_years = [y for y in years if 2019 <= y <= 2026]
    
    # Use A90 Supra default range if no valid years found
    if valid_years:
        model_year_min = min(valid_years)
        model_year_max = max(valid_years)
    else:
        model_year_min = 2019
        model_year_max = 2026

    car_fitment_flags = {
        'fits_gr_supra': bool(re.search(r'gr supra|mkv supra|mk5 supra|j29|a90|a91', t)),
        'fits_a90': 'a90' in t or 'j29' in t,
        'fits_a91': 'a91' in t,
        'fits_b58': 'b58' in t or '3.0' in t,
        'fits_b48': 'b48' in t or '2.0' in t,
        'fits_3_0': 'b58' in t or '3.0' in t,
        'fits_2_0': 'b48' in t or '2.0' in t,
        'model_year_min': model_year_min,
        'model_year_max': model_year_max,
    }
    
    return car_fitment_flags

def extract_numeric_claims(text: str) -> dict[str, Any]:
    t = text.lower()
    hp_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(whp|hp|horsepower)\b', t)]
    tq_vals = [int(v.replace('+', '')) for v, _ in re.findall(r'(\+?\d{1,4})\s?(tq|torque|lb[- ]?ft|ft[- ]?lb)\b', t)]
    weight_matches = re.findall(r'([+-]?\d+(?:\.\d+)?)\s?(lbs?|pounds|kg)\b', t)
    weights_lbs = []
    #Convert extracted value into pounds if in kg
    for val, unit in weight_matches:
        float_val = float(val)
        if unit == 'kg':
            float_val *= 2.20462
        weights_lbs.append(float_val)


    extracted_numeric_claims = {
        'claimed_hp_gain_min': min(hp_vals) if hp_vals else None,
        'claimed_hp_gain_max': max(hp_vals) if hp_vals else None,
        'claimed_tq_gain_min': min(tq_vals) if tq_vals else None,
        'claimed_tq_gain_max': max(tq_vals) if tq_vals else None,
        'weight_delta_lbs': weights_lbs[0] if weights_lbs else None,
        'hp_gain_source': 'stated' if hp_vals else 'missing',
    }

    return 


In [3]:
#risk_dependencies analyzes product text, category, and subsystem to infer installation and risk attributes.
def risk_dependencies(text: str, system_category: str, subsystem: str) -> dict[str, Any]:
    t = text.lower()
    requires_tune = any(keyword in text for keyword in ['requires tune', 'tune required', 'ecu tune', 'calibration required']) or subsystem in ['downpipe', 'turbo', 'fueling']
    requires_fueling = any(keyword in text for keyword in ['e85', 'ethanol', 'port injection', 'fuel pump', 'injector']) or subsystem == 'turbo'
    requires_cooling = any(keyword in text for keyword in ['heat exchanger', 'oil cooler', 'radiator', 'cooling']) or subsystem == 'turbo'
    requires_install = subsystem in ['turbo', 'fueling', 'suspension', 'brakes'] or any(keyword in text for keyword in ['professional installation', 'install by a professional'])
    
    if any(keyword in text for keyword in ['catless', 'off-road', 'race use only', 'not street legal']):
        emissions_risk = 'high'
    elif subsystem in ['downpipe', 'exhaust', 'tune'] or system_category in ['exhaust', 'tuning']:
        emissions_risk = 'medium'
    else:
        emissions_risk = 'low'
    if subsystem in ['turbo', 'fueling']:
        reliability_risk = 'high'
    elif subsystem in ['tune', 'downpipe', 'suspension', 'brakes']:
        reliability_risk = 'medium'
    else:
        reliability_risk = 'low'
    if subsystem in ['turbo', 'fueling']:
        install_complexity = 'high'
    elif subsystem in ['suspension', 'brakes', 'aero', 'downpipe']:
        install_complexity = 'medium'
    else:
        install_complexity = 'low'


    risk_dependencies ={
        'requires_tune': requires_tune,
        'requires_fueling': requires_fueling,
        'requires_cooling': requires_cooling,
        'requires_professional_install': requires_install,
        'emissions_risk': emissions_risk,
        'reliability_risk': reliability_risk,
        'install_complexity': install_complexity,
    }
    return risk_dependencies

In [4]:
CATEGORY_MAP = {
    'brake rotors': 'brakes', 'brakes': 'brakes', 'brake pads': 'brakes',
    'intakes': 'engine_power', 'intake': 'engine_power', 'intake filters': 'engine_power',
    'intake manifolds': 'engine_power', 'performance': 'engine_power',
    'performance accessories': 'engine_power', 'turbo': 'engine_power', 'turbos': 'engine_power',
    'turbo kits': 'engine_power', 'turbo kit': 'engine_power', 'bov': 'engine_power',
    'camshafts': 'engine_power', 'charge/boost pipes': 'engine_power',
    'downpipes': 'exhaust', 'exhaust': 'exhaust', 'exhausts': 'exhaust',
    'tuning': 'tuning', 'ecu': 'tuning', 'fueling': 'fueling',
    'port injection kit': 'fueling', 'flex fuel': 'fueling', 'cooling': 'cooling',
    'heat exchangers': 'cooling', 'suspension': 'suspension', 'coilovers': 'suspension',
    'springs': 'suspension', 'suspension components': 'suspension',
    'aero': 'aero', 'front lips/splitters': 'aero', 'spoilers & wings': 'aero',
    'rear skirts/spats': 'aero', 'side skirts': 'aero', 'wheels': 'wheels_tires',
    'tires': 'wheels_tires', 'maintenance': 'maintenance', 'electronics': 'electronics',
    'styling': 'styling', 'engine dress up': 'styling', 'body kits': 'styling',
    'mirrors': 'styling', 'steering wheel': 'styling', 'accessories': 'accessories',
}


PART_KEYWORDS = {
    'turbo': ['turbo', 'pure700', 'pure 700', 'g35', 'g42', 'precision', 'hybrid turbo'],
    'tune': ['tune', 'bootmod3', 'bm3', 'ecutek', 'mhd', 'motec', 'femto', 'ecu unlock'],
    'downpipe': ['downpipe', 'catless', 'catted dp', 'high-flow cat'],
    'intake': ['intake', 'airbox', 'snorkel', 'charge pipe', 'inlet'],
    'fueling': ['fuel pump', 'hpfp', 'lpfp', 'port injection', 'injector', 'e50', 'e85', 'ethanol'],
    'cooling': ['heat exchanger', 'radiator', 'oil cooler', 'cooling', 'intercooler'],
    'exhaust': ['exhaust', 'catback', 'straight pipe', 'muffler', 'titanium exhaust'],
    'brakes': ['brake', 'bbk', 'rotor', 'pad', 'caliper', 'alcon', 'brembo'],
    'suspension': ['coilover', 'spring', 'sway bar', 'control arm', 'camber', 'spl', 'mcs'],
    'aero': ['wing', 'splitter', 'diffuser', 'canard', 'verus', 'apr'],
    'wheels_tires': ['tire', 'tyre', 'wheel', 'apex', 'volk', 'te37', 'nankang', 'yokohama'],
}

RAW_DIR = Path('../backend/app/data/processed')
OUT_DIR = RAW_DIR / 'redlineiq_cleaned'
OUT_DIR.mkdir(exist_ok=True)

# ETL Pipelines

## Parts - Goal is to answer these questions:
- Does this part fit the A90 Supra?
- Does it fit B58 or B48?
- What category is it really?
- Does it add power?
- Does it require a tune?
- Does it require fueling?
- Does it increase emissions risk?
- Does it increase reliability risk?
- Is it useful for street, track, time attack, or drag?
- What evidence supports it?
- How confident is the system?

This data will be used for the Parts Intelligence

In [5]:
parts_df = pd.read_csv('../backend/app/data/ingestion/raw/parts_data_raw/final_categories_data.csv')
parts_df.head()
parts_df.describe()

,price,regular_price,sale_price
count,1664.000000,1664.000000,1664.000000
mean,1054.681929,1092.604207,1054.681929
std,1445.080110,1484.585596,1445.080110
min,2.990000,2.990000,2.990000
25%,206.020000,229.000000,206.020000
50%,550.000000,594.500000,550.000000
75%,1195.000000,1278.040000,1195.000000
max,11000.000000,11000.000000,11000.000000


In [6]:
print(parts_df.shape)
#check unique product_url
print(parts_df['product_url'].nunique())
#check for rows with duplicate product_url
duplicate_product_urls = parts_df[parts_df.duplicated(subset=['product_url'])]
print(duplicate_product_urls.shape)

(1664, 18)
706
(958, 18)


In [7]:
#filter out rows with duplicate product_url and keep the first occurrence
parts_df_unique = parts_df.drop_duplicates(subset=['product_url'], keep='first')
parts_df_unique.info()

<class 'pandas.DataFrame'>
Index: 706 entries, 0 to 1620
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   category           706 non-null    str    
 1   category_value     706 non-null    str    
 2   product_name       706 non-null    str    
 3   brand              706 non-null    str    
 4   sku                604 non-null    str    
 5   price              706 non-null    float64
 6   regular_price      706 non-null    float64
 7   sale_price         706 non-null    float64
 8   currency           706 non-null    str    
 9   description        706 non-null    str    
 10  specifications     134 non-null    str    
 11  fitment            148 non-null    str    
 12  features           703 non-null    str    
 13  image_urls         706 non-null    str    
 14  product_url        706 non-null    str    
 15  source_product_id  706 non-null    str    
 16  breadcrumb         705 non-null    str   

In [8]:
# range of price values and sale_price values
print("Price range: ", parts_df_unique['price'].min(), " - ", parts_df_unique['price'].max())
print("Sale Price range: ", parts_df_unique['sale_price'].min(), " - ", parts_df_unique['sale_price'].max())
print("Regular Price range: ", parts_df_unique['regular_price'].min(), " - ", parts_df_unique['regular_price'].max())

Price range:  2.99  -  11000.0
Sale Price range:  2.99  -  11000.0
Regular Price range:  2.99  -  11000.0


In [9]:
#Check correlation between price, regular_price, and sale_price
correlation_matrix = parts_df_unique[['price', 'regular_price', 'sale_price']].corr()
print(correlation_matrix)


                  price  regular_price  sale_price
price          1.000000       0.995983    1.000000
regular_price  0.995983       1.000000    0.995983
sale_price     1.000000       0.995983    1.000000


In [10]:
# Final Dataframe for analysis and modeling
parts_df_final = parts_df_unique[['product_name','category', 'brand', 'sku', 'price', 'regular_price', 'description', 'specifications', 'product_url']]

In [11]:
parts_df_final

,product_name,category,brand,sku,price,regular_price,description,specifications,product_url
0,R1 - E Line Blank Rotors w/ Ceramic Pads MKV S...,Brake Rotors,R1,WFWN1-31008,162.62,162.62,R1 Concepts - E Line Blank Rotors w/ Ceramic P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...
1,R1 - E Line Blank Rotors w/ OPTIMUM Pads MKV S...,Brake Rotors,R1,WFUN1-31766,189.97,189.97,R1 Concepts - E Line Blank Rotors w/ OPTIMUM P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...
2,R1 - E Line Drilled / Slotted Rotors w/ OPTIMU...,Brake Rotors,R1,WGUN1-31163,255.08,255.08,R1 Concepts - E Line Drilled / Slotted Rotors ...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...
3,R1 - E Line Black Drilled / Slotted Rotors w/ ...,Brake Rotors,R1,WHUN1-31168,325.63,325.63,R1 Concepts - E Line Black Drilled / Slotted R...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...
4,R1 - GEO Carbon Drilled / Slotted w/ Performan...,Brake Rotors,R1,WBSN1-31221,384.99,384.99,R1 Concepts - GEO Carbon Drilled / Slotted Rot...,and ensure proper fit and function.R1 - GEO Ca...,https://www.a90shop.com/product-page/r1-geo-ca...
...,...,...,...,...,...,...,...,...,...
1532,Dinan - Performance Spring Set MKV Supra A90,Suspension,Dinan,D100-0938,337.95,380.95,Dinan - Performance Spring Set MKV Supra A90 S...,NaN,https://www.a90shop.com/product-page/dinan-per...
1546,aFe Control Stage-1 Suspension Package 2020 To...,Suspension,aFe,510-721001-L,784.00,784.00,The iconic Supra is back! While the new Supra ...,NaN,https://www.a90shop.com/product-page/afe-contr...
1583,"aFe Control Stage-1 Suspension Package, MKV Supra",Suspension,aFe,afe510-721001-L,784.00,784.00,CNC Bent Sway Bars:Every aFe Control sway bar ...,"CNC Bent, Lightweight 1026 DOM Steel Tubular S...",https://www.a90shop.com/product-page/afe-contr...
1594,"Verus Engineering Front Camber Plate Assembly,...",Suspension,Verus,A0237AFrom,574.95,574.95,The Verus Engineering Camber Plates allow you ...,Billet 6061-T6 Machined Aluminum Construction ...,https://www.a90shop.com/product-page/verus-fro...


In [12]:
parts_df_final['fitment'] = 'A90 Supra'

In [13]:
parts_df_final.head()

,product_name,category,brand,sku,price,regular_price,description,specifications,product_url,fitment
0,R1 - E Line Blank Rotors w/ Ceramic Pads MKV S...,Brake Rotors,R1,WFWN1-31008,162.62,162.62,R1 Concepts - E Line Blank Rotors w/ Ceramic P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra
1,R1 - E Line Blank Rotors w/ OPTIMUM Pads MKV S...,Brake Rotors,R1,WFUN1-31766,189.97,189.97,R1 Concepts - E Line Blank Rotors w/ OPTIMUM P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra
2,R1 - E Line Drilled / Slotted Rotors w/ OPTIMU...,Brake Rotors,R1,WGUN1-31163,255.08,255.08,R1 Concepts - E Line Drilled / Slotted Rotors ...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra
3,R1 - E Line Black Drilled / Slotted Rotors w/ ...,Brake Rotors,R1,WHUN1-31168,325.63,325.63,R1 Concepts - E Line Black Drilled / Slotted R...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra
4,R1 - GEO Carbon Drilled / Slotted w/ Performan...,Brake Rotors,R1,WBSN1-31221,384.99,384.99,R1 Concepts - GEO Carbon Drilled / Slotted Rot...,and ensure proper fit and function.R1 - GEO Ca...,https://www.a90shop.com/product-page/r1-geo-ca...,A90 Supra


In [14]:
parts_df_final.category.unique()

<ArrowStringArray>
[           'Brake Rotors',          'Wind Deflector',
             'Accessories',                     'BOV',
      'Port Injection Kit',       'Rear Skirts/Spats',
              'Brake Pads',                   'Hoods',
     'Manual Transmission',                 'Intakes',
           'Front Fenders',               'Flex Fuel',
    'Single Exit Exhausts',              'Strut Bars',
             'Electronics',               'Diffusers',
         'Heat Exchangers',           'Shift Paddles',
       'Wheel Accessories',                  'Brakes',
               'Body Kits', 'Performance Accessories',
               'Downpipes',                    'Aero',
          'Intake Filters',               'Sway Bars',
               'Camshafts',                 'Mirrors',
                  'Wheels',                 'Springs',
        'Spoilers & Wings',                  'Turbos',
               'Turbo Kit',             'Side Skirts',
                'Exhausts',                   

In [15]:
parts_df_final['combined_text'] = (
        parts_df_final['brand'].fillna('') + ' ' + parts_df_final['product_name'].fillna('') + ' ' +
        parts_df_final['category'].fillna('') + ' ' + parts_df_final['description'].fillna('') + ' ' +
        parts_df_final['specifications'].fillna('') 
    ).str.strip()

In [16]:
parts_df_final['system_category'] = parts_df_final.apply(lambda x: normalize_system_category(x['category'], x['combined_text']), axis=1)
parts_df_final['subsystem'] = parts_df_final['combined_text'].apply(extract_subsystem)

In [17]:
parts_df_final

,product_name,category,brand,sku,price,regular_price,description,specifications,product_url,fitment,combined_text,system_category,subsystem
0,R1 - E Line Blank Rotors w/ Ceramic Pads MKV S...,Brake Rotors,R1,WFWN1-31008,162.62,162.62,R1 Concepts - E Line Blank Rotors w/ Ceramic P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,R1 R1 - E Line Blank Rotors w/ Ceramic Pads MK...,brakes,brakes
1,R1 - E Line Blank Rotors w/ OPTIMUM Pads MKV S...,Brake Rotors,R1,WFUN1-31766,189.97,189.97,R1 Concepts - E Line Blank Rotors w/ OPTIMUM P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,R1 R1 - E Line Blank Rotors w/ OPTIMUM Pads MK...,brakes,brakes
2,R1 - E Line Drilled / Slotted Rotors w/ OPTIMU...,Brake Rotors,R1,WGUN1-31163,255.08,255.08,R1 Concepts - E Line Drilled / Slotted Rotors ...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,R1 R1 - E Line Drilled / Slotted Rotors w/ OPT...,brakes,brakes
3,R1 - E Line Black Drilled / Slotted Rotors w/ ...,Brake Rotors,R1,WHUN1-31168,325.63,325.63,R1 Concepts - E Line Black Drilled / Slotted R...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,R1 R1 - E Line Black Drilled / Slotted Rotors ...,brakes,brakes
4,R1 - GEO Carbon Drilled / Slotted w/ Performan...,Brake Rotors,R1,WBSN1-31221,384.99,384.99,R1 Concepts - GEO Carbon Drilled / Slotted Rot...,and ensure proper fit and function.R1 - GEO Ca...,https://www.a90shop.com/product-page/r1-geo-ca...,A90 Supra,R1 R1 - GEO Carbon Drilled / Slotted w/ Perfor...,brakes,brakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532,Dinan - Performance Spring Set MKV Supra A90,Suspension,Dinan,D100-0938,337.95,380.95,Dinan - Performance Spring Set MKV Supra A90 S...,NaN,https://www.a90shop.com/product-page/dinan-per...,A90 Supra,Dinan Dinan - Performance Spring Set MKV Supra...,suspension,suspension
1546,aFe Control Stage-1 Suspension Package 2020 To...,Suspension,aFe,510-721001-L,784.00,784.00,The iconic Supra is back! While the new Supra ...,NaN,https://www.a90shop.com/product-page/afe-contr...,A90 Supra,aFe aFe Control Stage-1 Suspension Package 202...,suspension,turbo
1583,"aFe Control Stage-1 Suspension Package, MKV Supra",Suspension,aFe,afe510-721001-L,784.00,784.00,CNC Bent Sway Bars:Every aFe Control sway bar ...,"CNC Bent, Lightweight 1026 DOM Steel Tubular S...",https://www.a90shop.com/product-page/afe-contr...,A90 Supra,"aFe aFe Control Stage-1 Suspension Package, MK...",suspension,turbo
1594,"Verus Engineering Front Camber Plate Assembly,...",Suspension,Verus,A0237AFrom,574.95,574.95,The Verus Engineering Camber Plates allow you ...,Billet 6061-T6 Machined Aluminum Construction ...,https://www.a90shop.com/product-page/verus-fro...,A90 Supra,Verus Verus Engineering Front Camber Plate Ass...,suspension,downpipe


In [18]:
fit = parts_df_final['combined_text'].apply(extract_fitment_flags).apply(pd.Series)

fit

,fits_gr_supra,fits_a90,fits_a91,fits_b58,fits_b48,fits_3_0,fits_2_0,model_year_min,model_year_max
0,True,True,False,False,True,False,True,2019,2026
1,True,True,False,False,True,False,True,2019,2026
2,True,True,False,False,True,False,True,2019,2026
3,True,True,False,False,True,False,True,2019,2026
4,True,True,False,False,True,False,True,2019,2026
...,...,...,...,...,...,...,...,...,...
1532,True,True,False,False,False,False,False,2019,2026
1546,True,True,False,True,False,True,False,2020,2020
1583,True,True,True,True,True,True,True,2020,2020
1594,True,True,True,True,False,True,False,2019,2026


In [19]:
nums = parts_df_final['combined_text'].apply(extract_numeric_claims).apply(pd.Series).fillna(0)

In [20]:

risk = parts_df_final.apply(lambda r: risk_dependencies(r['combined_text'], r['system_category'], r['subsystem']), axis=1).apply(pd.Series)
risk

,requires_tune,requires_fueling,requires_cooling,requires_professional_install,emissions_risk,reliability_risk,install_complexity
0,False,False,False,True,low,medium,medium
1,False,False,False,True,low,medium,medium
2,False,False,False,True,low,medium,medium
3,False,False,False,True,low,medium,medium
4,False,False,False,True,low,medium,medium
...,...,...,...,...,...,...,...
1532,False,False,False,True,low,medium,medium
1546,True,True,True,True,low,high,high
1583,True,True,True,True,low,high,high
1594,True,False,False,False,medium,medium,medium


In [21]:
products = pd.concat([parts_df_final, fit, nums, risk], axis=1)

In [22]:
products['discount_pct'] = np.where(products['regular_price'] > 0, (products['regular_price'] - products['price']) / products['regular_price'], 0)
products['price_bucket'] = pd.cut(products['price'], bins=[-np.inf, 500, 2000, np.inf], labels=['budget', 'mid', 'premium'])

# products['data_quality_score'] = products.apply(lambda r: data_quality_score(r, list(products.columns)), axis=1)


In [23]:
products

,product_name,category,brand,sku,price,regular_price,description,specifications,product_url,fitment,...,model_year_max,requires_tune,requires_fueling,requires_cooling,requires_professional_install,emissions_risk,reliability_risk,install_complexity,discount_pct,price_bucket
0,R1 - E Line Blank Rotors w/ Ceramic Pads MKV S...,Brake Rotors,R1,WFWN1-31008,162.62,162.62,R1 Concepts - E Line Blank Rotors w/ Ceramic P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.000000,budget
1,R1 - E Line Blank Rotors w/ OPTIMUM Pads MKV S...,Brake Rotors,R1,WFUN1-31766,189.97,189.97,R1 Concepts - E Line Blank Rotors w/ OPTIMUM P...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.000000,budget
2,R1 - E Line Drilled / Slotted Rotors w/ OPTIMU...,Brake Rotors,R1,WGUN1-31163,255.08,255.08,R1 Concepts - E Line Drilled / Slotted Rotors ...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.000000,budget
3,R1 - E Line Black Drilled / Slotted Rotors w/ ...,Brake Rotors,R1,WHUN1-31168,325.63,325.63,R1 Concepts - E Line Black Drilled / Slotted R...,and ensure proper fit and function.R1 - E Line...,https://www.a90shop.com/product-page/r1-e-line...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.000000,budget
4,R1 - GEO Carbon Drilled / Slotted w/ Performan...,Brake Rotors,R1,WBSN1-31221,384.99,384.99,R1 Concepts - GEO Carbon Drilled / Slotted Rot...,and ensure proper fit and function.R1 - GEO Ca...,https://www.a90shop.com/product-page/r1-geo-ca...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.000000,budget
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1532,Dinan - Performance Spring Set MKV Supra A90,Suspension,Dinan,D100-0938,337.95,380.95,Dinan - Performance Spring Set MKV Supra A90 S...,NaN,https://www.a90shop.com/product-page/dinan-per...,A90 Supra,...,2026,False,False,False,True,low,medium,medium,0.112876,budget
1546,aFe Control Stage-1 Suspension Package 2020 To...,Suspension,aFe,510-721001-L,784.00,784.00,The iconic Supra is back! While the new Supra ...,NaN,https://www.a90shop.com/product-page/afe-contr...,A90 Supra,...,2020,True,True,True,True,low,high,high,0.000000,mid
1583,"aFe Control Stage-1 Suspension Package, MKV Supra",Suspension,aFe,afe510-721001-L,784.00,784.00,CNC Bent Sway Bars:Every aFe Control sway bar ...,"CNC Bent, Lightweight 1026 DOM Steel Tubular S...",https://www.a90shop.com/product-page/afe-contr...,A90 Supra,...,2020,True,True,True,True,low,high,high,0.000000,mid
1594,"Verus Engineering Front Camber Plate Assembly,...",Suspension,Verus,A0237AFrom,574.95,574.95,The Verus Engineering Camber Plates allow you ...,Billet 6061-T6 Machined Aluminum Construction ...,https://www.a90shop.com/product-page/verus-fro...,A90 Supra,...,2026,True,False,False,False,medium,medium,medium,0.000000,mid


In [24]:
products.info()

<class 'pandas.DataFrame'>
Index: 706 entries, 0 to 1620
Data columns (total 31 columns):
 #   Column                         Non-Null Count  Dtype   
---  ------                         --------------  -----   
 0   product_name                   706 non-null    str     
 1   category                       706 non-null    str     
 2   brand                          706 non-null    str     
 3   sku                            604 non-null    str     
 4   price                          706 non-null    float64 
 5   regular_price                  706 non-null    float64 
 6   description                    706 non-null    str     
 7   specifications                 134 non-null    str     
 8   product_url                    706 non-null    str     
 9   fitment                        706 non-null    str     
 10  combined_text                  706 non-null    str     
 11  system_category                706 non-null    str     
 12  subsystem                      706 non-null    str 

In [25]:

chunks = products[['product_url', 'product_name', 'combined_text']].copy()
chunks['chunk_id'] = chunks.apply(lambda r: stable_id(r['product_url'], 'product_text', prefix='chunk_'), axis=1)
chunks['source_type'] = 'product_catalog'
chunks['chunk_type'] = 'description_spec_fitment'
chunks = chunks[['chunk_id', 'source_type', 'chunk_type', 'product_url', 'product_name', 'combined_text']]


In [26]:
chunks

,chunk_id,source_type,chunk_type,product_url,product_name,combined_text
0,chunk_39eb74769e86,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/r1-e-line...,R1 - E Line Blank Rotors w/ Ceramic Pads MKV S...,R1 R1 - E Line Blank Rotors w/ Ceramic Pads MK...
1,chunk_84a7feff77e4,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/r1-e-line...,R1 - E Line Blank Rotors w/ OPTIMUM Pads MKV S...,R1 R1 - E Line Blank Rotors w/ OPTIMUM Pads MK...
2,chunk_9fe8ee56dfe6,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/r1-e-line...,R1 - E Line Drilled / Slotted Rotors w/ OPTIMU...,R1 R1 - E Line Drilled / Slotted Rotors w/ OPT...
3,chunk_601f265a823b,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/r1-e-line...,R1 - E Line Black Drilled / Slotted Rotors w/ ...,R1 R1 - E Line Black Drilled / Slotted Rotors ...
4,chunk_244e43851c38,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/r1-geo-ca...,R1 - GEO Carbon Drilled / Slotted w/ Performan...,R1 R1 - GEO Carbon Drilled / Slotted w/ Perfor...
...,...,...,...,...,...,...
1532,chunk_33ee6c093e14,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/dinan-per...,Dinan - Performance Spring Set MKV Supra A90,Dinan Dinan - Performance Spring Set MKV Supra...
1546,chunk_b7392dbfe6e3,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/afe-contr...,aFe Control Stage-1 Suspension Package 2020 To...,aFe aFe Control Stage-1 Suspension Package 202...
1583,chunk_f27e00619e23,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/afe-contr...,"aFe Control Stage-1 Suspension Package, MKV Supra","aFe aFe Control Stage-1 Suspension Package, MK..."
1594,chunk_8b41767d8649,product_catalog,description_spec_fitment,https://www.a90shop.com/product-page/verus-fro...,"Verus Engineering Front Camber Plate Assembly,...",Verus Verus Engineering Front Camber Plate Ass...


In [27]:
categories = parts_df[[ 'product_url', 'category', 'category_value']].drop_duplicates().copy()
categories['normalized_category'] = categories.apply(lambda r: normalize_system_category(r['category'], str(r['category'])), axis=1)

categories

,product_url,category,category_value,normalized_category
0,https://www.a90shop.com/product-page/r1-e-line...,Brake Rotors,6c840391-47ec-081b-e8a4-718824696e40,brakes
1,https://www.a90shop.com/product-page/r1-e-line...,Brake Rotors,6c840391-47ec-081b-e8a4-718824696e40,brakes
2,https://www.a90shop.com/product-page/r1-e-line...,Brake Rotors,6c840391-47ec-081b-e8a4-718824696e40,brakes
3,https://www.a90shop.com/product-page/r1-e-line...,Brake Rotors,6c840391-47ec-081b-e8a4-718824696e40,brakes
4,https://www.a90shop.com/product-page/r1-geo-ca...,Brake Rotors,6c840391-47ec-081b-e8a4-718824696e40,brakes
...,...,...,...,...
1659,https://www.a90shop.com/product-page/verus-eng...,Front Lips/Splitters,39da7b5b-0676-fb88-33f1-1786ebbf2ea1,aero
1660,https://www.a90shop.com/product-page/rexpeed-v...,Front Lips/Splitters,39da7b5b-0676-fb88-33f1-1786ebbf2ea1,aero
1661,https://www.a90shop.com/product-page/rexpeed-m...,Front Lips/Splitters,39da7b5b-0676-fb88-33f1-1786ebbf2ea1,aero
1662,https://www.a90shop.com/product-page/rexpeed-m...,Front Lips/Splitters,39da7b5b-0676-fb88-33f1-1786ebbf2ea1,aero


In [29]:
products.to_csv(OUT_DIR / 'products_clean.csv', index=False)
categories.to_csv(OUT_DIR / 'product_categories_clean.csv', index=False)
chunks.to_csv(OUT_DIR / 'product_evidence_chunks.csv', index=False)